In [1]:
# Let's install and import Pydantic
# In Pydantic, BaseModel is the core class that you use to create data models.
# BaseModel is like a blueprint for structured data. It defines the fields, their types, and automatically gives you data validation and type conversion capabilities
# !pip install pydantic
from pydantic import BaseModel

In [2]:
# Install necessary libraries if running for the first time
# !pip install openai google-generativeai python-dotenv ipython

# Import necessary libraries
import os
import google.generativeai as genai
from openai import OpenAI  # Make sure you have the latest openai package (pip install --upgrade openai)
from dotenv import load_dotenv
import json

#  Importing type hints that help describe what kind of data your Python functions or classes expect or return.
# List: A list of elements, all usually of the same type.
# Example: List[int] means a list of integers like [1, 2, 3].

# Dict: A dictionary (key-value pairs).
# Example: Dict[str, int] means keys are strings and values are integers like {'a': 1, 'b': 2}.

# Union: Either one type or another.
# Example: Union[int, str] means the value can be an int or a str.

# Optional: Means a value can be the type you expect or None.
# Example: Optional[int] is the same as Union[int, None].

# Any: Anything at all — no restriction on type.
# You can pass an int, string, list, object, etc.
from typing import List, Dict, Union, Optional, Any
from IPython.display import display, Markdown

print("Libraries imported successfully!")

# Load environment variables from the .env file
load_dotenv()

# Fetch API keys from environment variables
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")


# Configure the APIs
openai_client = OpenAI(api_key = openai_api_key)
genai.configure(api_key = google_api_key)

# Initialize the Gemini model, choose a suitable model like "gemini-2.0-flash"
gemini_model = genai.GenerativeModel("gemini-2.0-flash")


Libraries imported successfully!


/Users/asmi/workspace/AI-CausalGraph-Driven-HRProcessOptimizer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Helper function to display markdown nicely 
def print_markdown(text):
    """Displays text as Markdown."""
    display(Markdown(text))

In [4]:
# Let's define a sample resume text
resume_text = """
**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Marketing professsional with 2 years of experience assisting in digital campaigns, content creation, and social media activities. Comfortable handling multiple tasks and providing general marketing support.

**Experience**

**Marketing Asssistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted with digital marketing campaigns including email and social media.
- Created blog posts and social media updates to improve audience engagement.
- Managed social media accounts and grew follower numbers.
- Supported coordination of marketing events.
- Conducted market research and competitor analysis.

**Skils**
- Digital Marketing (SEO basics, Email Marketing)
- Social Media Tools (Hootsuite, Buffer)
- Microsoft Office Suite, Google Workspace
- Basic knowledge of Adobe Photoshop

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021
"""


In [5]:
# Let's define a sample job description text
job_description_text = """
# Job Title: Digital Marketing Specialist

**Company:** BrightWave Digital Agency

**Location:** Toronto, ON

## About Us:
BrightWave Digital Agency creates digital marketing campaigns for a variety of clients. We are looking for a Digital Marketing Specialist to join our team and assist in managing campaigns.

## Responsibilities:
- Assist in planning and executing digital marketing campaigns (SEO, SEM, social media, email).
- Use Google Analytics to measure performance and prepare basic performance reports.
- Support social media management tasks including content scheduling and community engagement.
- Perform keyword research and assist in optimizing content for SEO.
- Work with designers to help coordinate campaign materials.
- Keep informed about current digital marketing trends.

## Qualifications:
- Bachelor's degree in Marketing, Communications, or similar.
- 2+ years of digital marketing experience.
- Familiarity with SEO, SEM, Google Analytics, and social media.
- Ability to interpret basic marketing data.
- Good communication and writing skills.
- Knowledge of CRM systems (e.g., HubSpot) helpful.
- Experience with Adobe Creative Suite is beneficial.
"""

In [6]:
# Let's display the original resume 
print_markdown("**--- Original Resume ---**")
print_markdown(resume_text)

**--- Original Resume ---**


**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Marketing professsional with 2 years of experience assisting in digital campaigns, content creation, and social media activities. Comfortable handling multiple tasks and providing general marketing support.

**Experience**

**Marketing Asssistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted with digital marketing campaigns including email and social media.
- Created blog posts and social media updates to improve audience engagement.
- Managed social media accounts and grew follower numbers.
- Supported coordination of marketing events.
- Conducted market research and competitor analysis.

**Skils**
- Digital Marketing (SEO basics, Email Marketing)
- Social Media Tools (Hootsuite, Buffer)
- Microsoft Office Suite, Google Workspace
- Basic knowledge of Adobe Photoshop

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021


In [7]:
# Let's display the target job desciption
print_markdown("\n**--- Target Job Description ---**")
print_markdown(job_description_text)


**--- Target Job Description ---**


# Job Title: Digital Marketing Specialist

**Company:** BrightWave Digital Agency

**Location:** Toronto, ON

## About Us:
BrightWave Digital Agency creates digital marketing campaigns for a variety of clients. We are looking for a Digital Marketing Specialist to join our team and assist in managing campaigns.

## Responsibilities:
- Assist in planning and executing digital marketing campaigns (SEO, SEM, social media, email).
- Use Google Analytics to measure performance and prepare basic performance reports.
- Support social media management tasks including content scheduling and community engagement.
- Perform keyword research and assist in optimizing content for SEO.
- Work with designers to help coordinate campaign materials.
- Keep informed about current digital marketing trends.

## Qualifications:
- Bachelor's degree in Marketing, Communications, or similar.
- 2+ years of digital marketing experience.
- Familiarity with SEO, SEM, Google Analytics, and social media.
- Ability to interpret basic marketing data.
- Good communication and writing skills.
- Knowledge of CRM systems (e.g., HubSpot) helpful.
- Experience with Adobe Creative Suite is beneficial.


Now that we have our resume and job description, let's use the OpenAI API to improve the resume to better match the job requirements. We'll make a call to the text generation API and ask it to enhance our resume.


In [8]:
def openai_generate(prompt: str,
                    model: str = "gpt-4o",
                    temperature: float = 0.7,
                    max_tokens: int = 1500,
                    response_format: Optional[dict] = None) -> str | dict:
    """
    Generate text using OpenAI API

    This function sends a prompt to OpenAI's API and returns the generated response.
    It supports both standard text generation and structured parsing with response_format.

    Args:
        prompt (str): The prompt to send to the model, i.e.: your instructions for the AI
        model (str): The OpenAI model to use (default: "gpt-4o")
        temperature (float): Controls randomness, where lower values make output more deterministic
        max_tokens (int): Maximum number of tokens to generate, which limits the response length
        response_format (dict): Optional format specification
        In simple terms, response_format is optional. If the user gives me a dictionary, cool! 
        If they don't give me anything, just assume it's None and keep going."

    Returns:
        str or dict: The generated text or parsed structured data, depending on response_format
    """

    
    try:
        # Standard text generation without a specific response format
        if not response_format:
            response = openai_client.chat.completions.create(
                model = model,
                messages = [
                    {"role": "system",
                     "content": "You are a helpful assistant specializing in resume writing and career advice.",
                    },
                    {"role": "user", "content": prompt}],
                temperature = temperature,
                max_tokens = max_tokens)
            
            # Extract just the text content from the response
            return response.choices[0].message.content
        
        # Structured response generation (e.g., JSON format)
        else:
            completion = openai_client.beta.chat.completions.parse(
                model = model,  # Make sure to use a model that supports parse
                messages = [
                    # Same system and user messages as above
                    {
                        "role": "system",
                        "content": "You are a helpful assistant specializing in resume writing and career advice.",
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature = temperature,
                response_format = response_format)

            # Return the parsed structured output
            return completion.choices[0].message.parsed
            
    except Exception as e:
        # Error handling to prevent crashes
        return f"Error generating text: {e}"


In [9]:
# Prompt to analyze the resume against the job description

def analyze_resume_against_job_description(job_description_text: str, resume_text: str, model: str = "openai") -> str:
    """
    Analyze the resume against the job description and return a structured comparison.

    Args:
        job_description_text (str): The job description text.
        resume_text (str): The candidate's resume text.
        model (str): The model to use for analysis ("openai" or "gemini").

    Returns:
        str: A clear, structured comparison of the resume and job description.
    """
    # This prompt instructs the AI to act as a career advisor and analyze how well the resume matches the job description
    # It asks for a structured analysis with 4 specific sections: requirements, matches, gaps, and strengths
    prompt = f"""
    Context:
    You are a career advisor and resume expert. Your task is to analyze a candidate's resume against a specific job description to assess alignment and identify areas for improvement.

    Instruction:
    Review the provided Job Description and Resume. Identify key skills, experiences, and qualifications in the Job Description and compare them to what's present in the Resume. Provide a structured analysis with the following sections:
    1. **Key Requirements from Job Description:** List the main skills, experiences, and qualifications sought by the employer.
    2. **Relevant Experience in Resume:** List the skills and experiences from the resume that match or align closely with the job requirements.
    3. **Gaps/Mismatches:** Identify important skills or qualifications from the Job Description that are missing, unclear, or underrepresented in the Resume.
    4. **Potential Strengths:** Highlight any valuable skills, experiences, or accomplishments in the resume that are not explicitly requested in the job description but could strengthen the application.

    Job Description:

    {job_description_text}

    Resume:

    {resume_text}

    Output:
    Return a clear, structured comparison with the four sections outlined above.
    """

    # This conditional block selects which AI model to use based on the 'model' parameter
    if model == "openai":
        # Uses OpenAI's model to generate the gap analysis with moderate creativity (temperature=0.7)
        gap_analysis = openai_generate(prompt, temperature=0.7)
    elif model == "gemini":
        # Uses Google's Gemini model with less creativity (temperature=0.5) for more focused results
        gap_analysis = gemini_generate(prompt, temperature=0.5)
    else:
        # Raises an error if an invalid model name is provided
        raise ValueError(f"Invalid model: {model}")

    # Returns the generated gap analysis text
    return gap_analysis



In [10]:
# Call the function to analyze the resume against the job description using OpenAI
gap_analysis_openai = analyze_resume_against_job_description(job_description_text, 
                                                             resume_text, 
                                                             model = "openai")

# Displays the analysis results in Markdown format
print_markdown("#### OpenAI Response:")
print_markdown(gap_analysis_openai)

#### OpenAI Response:

### 1. Key Requirements from Job Description:

- **Experience and Skills:**
  - Assist in planning and executing digital marketing campaigns (SEO, SEM, social media, email).
  - Use Google Analytics to measure performance and prepare basic performance reports.
  - Support social media management tasks including content scheduling and community engagement.
  - Perform keyword research and assist in optimizing content for SEO.
  - Work with designers to coordinate campaign materials.
  - Stay informed about current digital marketing trends.

- **Qualifications:**
  - Bachelor's degree in Marketing, Communications, or similar.
  - 2+ years of digital marketing experience.
  - Familiarity with SEO, SEM, Google Analytics, and social media.
  - Ability to interpret basic marketing data.
  - Good communication and writing skills.
  - Knowledge of CRM systems (e.g., HubSpot) is helpful.
  - Experience with Adobe Creative Suite is beneficial.

### 2. Relevant Experience in Resume:

- **Experience and Skills:**
  - Assisted with digital marketing campaigns, specifically in email and social media, aligning with the requirement to assist in campaign execution.
  - Managed social media accounts and increased follower numbers, which supports social media management tasks.
  - Created content (blog posts and social media updates) to improve audience engagement, relevant to community engagement.
  - Basic knowledge of SEO and email marketing, aligning with some digital marketing campaign requirements.

- **Qualifications:**
  - Bachelor of Commerce in Marketing, satisfying the educational requirement.
  - 2 years of experience as a Marketing Assistant, closely matching the required experience level.
  - Familiarity with social media tools like Hootsuite and Buffer.

### 3. Gaps/Mismatches:

- **Experience and Skills:**
  - No mention of SEM experience, which is part of the job requirements.
  - The resume does not specify experience with Google Analytics, which is crucial for performance measurement.
  - No evidence of conducting keyword research or optimizing content for SEO beyond basic knowledge.
  - Lack of explicit experience or collaboration with designers on campaign materials.

- **Qualifications:**
  - No mention of CRM system knowledge, such as HubSpot, which is noted as helpful.
  - Limited mention of Adobe Creative Suite experience, only basic Photoshop knowledge is listed.

### 4. Potential Strengths:

- Demonstrated ability to create engaging content and increase social media followings, which shows initiative and effective management of social media duties.
- Experience in conducting market research and competitor analysis, which could be beneficial for strategic planning and understanding the competitive landscape.
- Proficient in social media tools (Hootsuite, Buffer), which can enhance social media management tasks.
- Strong foundational knowledge in digital marketing areas (SEO basics, email marketing) that can be built upon with further training and experience.

**Recommendations for Improvement:**
- Gain experience or training in SEM and Google Analytics to fill critical gaps.
- Highlight any experience or coursework related to keyword research and SEO optimization.
- Consider gaining familiarity with CRM systems to enhance resume alignment with the job description.
- Explore additional experience or training in Adobe Creative Suite to strengthen design collaboration skills.

In [11]:
# Define Pydantic models for structured output
# The ResumeOutput class is a Pydantic model that defines the structure of the output
# for the resume generation function. It includes two fields:
# (1) updated_resume: A string that contains the final rewritten resume.
# (2) diff_markdown: A string containing the resume's HTML-coloured version highlighting additions and deletions.

class ResumeOutput(BaseModel):
    updated_resume: str
    diff_markdown: str


def generate_resume(
    job_description_text: str, resume_text: str, gap_analysis_openai: str, model: str = "openai") -> dict:
    """
    Generate a tailored resume using OpenAI or Gemini.

    Args:
        job_description_text (str): The job description text.
        resume_text (str): The candidate's resume text.
        gap_analysis_openai (str): The gap analysis result from OpenAI.
        model (str): The model to use for resume generation.

    Returns:
        dict: A dictionary containing the updated resume and diff markdown.
    """
    # Construct the prompt for the AI model to generate the tailored resume.
    # The prompt includes context, instructions, and input data (original resume,
    # target job description, and gap analysis).
    prompt = (
        """
    ### Context:
    You are an expert resume writer and editor. Your goal is to rewrite the original resume to match the target job description, using the provided tailoring suggestions and analysis.

    ---

    ### Instruction:
    1. Rewrite the entire resume to best match the **Target Job Description** and **Gap Analysis to the Job Description**.
    2. Improve clarity, add job-relevant keywords, and quantify achievements.
    3. Specifically address the gaps identified in the analysis by:
       - Adding missing skills and technologies mentioned in the job description
       - Reframing experience to highlight relevant accomplishments
       - Strengthening sections that were identified as weak in the analysis
    4. Prioritize addressing the most critical gaps first
    5. Incorporate industry-specific terminology from the job description
    6. Ensure all quantifiable achievements are properly highlighted with metrics
    7. Return two versions of the resume:
        - `updated_resume`: The final rewritten resume (as plain text)
        - `diff_html`: A version of the resume with inline highlights using color:
            - Additions or rewritten content should be **green**:  
            `<span style="color:green">your added or changed text</span>`
            - Removed content should be **red and struck through**:  
            `<span style="color:red;text-decoration:line-through">removed text</span>`
            - Leave unchanged lines unmarked.
        - Keep all section headers and formatting consistent with the original resume.

    ---

    ### Output Format:

    ```json
    {
    "updated_resume": "<full rewritten resume as plain text>",
    "diff_markdown": "<HTML-colored version of the resume highlighting additions and deletions>"
    }
    ```
    ---
    ### Input:

    **Original Resume:**

    """
        + resume_text
        + """


    **Target Job Description:**

    """
        + job_description_text
        + """


    **Analysis of Resume vs. Job Description:**

    """
        + gap_analysis_openai
    )

    # Depending on the selected model, call the appropriate function to generate the resume.
    # If the OpenAI model is selected, it uses a temperature of 0.7 for creativity.
    if model == "openai":
        updated_resume_json = openai_generate(prompt, temperature = 0.7, response_format = ResumeOutput)
    # If the Gemini model is selected, it uses a lower temperature of 0.5 for more focused results.
    elif model == "gemini":
        updated_resume_json = gemini_generate(prompt, temperature = 0.5)
    else:
        # Raise an error if an invalid model name is provided.
        raise ValueError(f"Invalid model: {model}")

    # Return the generated resume output as a dictionary.
    return updated_resume_json


In [12]:
# Call the generate_resume function with the provided job description, resume text, and gap analysis.
updated_resume_json = generate_resume(job_description_text, resume_text, gap_analysis_openai, model="openai")
# Display the updated resume in Markdown format.
print_markdown(updated_resume_json.updated_resume)

**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Dynamic digital marketing professional with over 2 years of experience in executing comprehensive digital marketing campaigns. Skilled in SEO, social media management, and email marketing with a proven track record of leveraging Google Analytics for performance measurement. Adept at keyword research and content optimization to enhance engagement and drive growth.

**Experience**

**Digital Marketing Specialist | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted in planning and executing digital marketing campaigns across SEO, SEM, social media, and email platforms, achieving a 30% increase in online engagement.
- Utilized Google Analytics to prepare performance reports, leading to a 15% improvement in campaign ROI through data-driven insights.
- Conducted keyword research and optimized content for SEO, increasing search engine visibility by 20%.
- Managed social media accounts, scheduling content and driving community engagement to grow followers by 25%.
- Collaborated with designers to coordinate campaign materials, ensuring consistent brand messaging.
- Stayed updated on current digital marketing trends to implement best practices.

**Skills**
- Digital Marketing: SEO, SEM, Email Marketing, Google Analytics
- Social Media Management: Hootsuite, Buffer
- Adobe Creative Suite: Photoshop, Illustrator
- CRM Systems: Basic knowledge of HubSpot
- Microsoft Office Suite, Google Workspace

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021

In [13]:
print_markdown(updated_resume_json.diff_markdown)

<span style="color:green">**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Dynamic digital marketing professional with over 2 years of experience in executing comprehensive digital marketing campaigns. Skilled in SEO, social media management, and email marketing with a proven track record of leveraging Google Analytics for performance measurement. Adept at keyword research and content optimization to enhance engagement and drive growth.

**Experience**

**Digital Marketing Specialist | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted in planning and executing digital marketing campaigns across SEO, SEM, social media, and email platforms, achieving a 30% increase in online engagement.
- Utilized Google Analytics to prepare performance reports, leading to a 15% improvement in campaign ROI through data-driven insights.
- Conducted keyword research and optimized content for SEO, increasing search engine visibility by 20%.
- Managed social media accounts, scheduling content and driving community engagement to grow followers by 25%.
- Collaborated with designers to coordinate campaign materials, ensuring consistent brand messaging.
- Stayed updated on current digital marketing trends to implement best practices.

**Skills**
- Digital Marketing: SEO, SEM, Email Marketing, Google Analytics
- Social Media Management: Hootsuite, Buffer
- Adobe Creative Suite: Photoshop, Illustrator
- CRM Systems: Basic knowledge of HubSpot
- Microsoft Office Suite, Google Workspace

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021</span>

<span style="color:red;text-decoration:line-through">Marketing professsional with 2 years of experience assisting in digital campaigns, content creation, and social media activities. Comfortable handling multiple tasks and providing general marketing support.

**Experience**

**Marketing Asssistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted with digital marketing campaigns including email and social media.
- Created blog posts and social media updates to improve audience engagement.
- Managed social media accounts and grew follower numbers.
- Supported coordination of marketing events.
- Conducted market research and competitor analysis.

**Skils**
- Digital Marketing (SEO basics, Email Marketing)
- Social Media Tools (Hootsuite, Buffer)
- Microsoft Office Suite, Google Workspace
- Basic knowledge of Adobe Photoshop

</span>

With the newly tailored resume, let's generate a corresponding cover letter. 
We'll use OpenAI, feeding it the tailored resume (from the previous task) and the original job description. This ensures the cover letter highlights the most relevant points from the improved resume.

In [14]:
# Define Pydantic models for structured output
# The CoverLetterOutput class is a Pydantic model that defines the structure of the output for the cover letter generation.
# It ensures that the output will contain a single field, 'cover_letter', which is a string.

class CoverLetterOutput(BaseModel):
    cover_letter: str

# The generate_cover_letter function creates a cover letter based on the provided job description and updated resume.
# It takes three parameters:
# (1) job_description_text: A string containing the job description for the position.
# (2) updated_resume: A string containing the candidate's updated resume.
# (3) model: A string indicating which model to use for generating the cover letter (default is "openai").
# The function returns a dictionary containing the generated cover letter.

def generate_cover_letter(job_description_text: str, updated_resume: str, model: str = "openai") -> dict:
    """
    Generate a cover letter using OpenAI or Gemini.

    Args:
        job_description_text (str): The job description text.
        updated_resume (str): The candidate's updated resume text.
        model (str): The model to use for cover letter generation.

    Returns:
        dict: A dictionary containing the cover letter.
    """

    # Construct the prompt for the AI model, including context and instructions for writing the cover letter.
    prompt = (
        """
    ### Context:
    You are a professional career coach and expert cover letter writer.

    ---

    ### Instruction:
    Write a compelling, personalized cover letter based on the **Updated Resume** and the **Target Job Description**. The letter should:
    1. Be addressed generically (e.g., "Dear Hiring Manager")
    2. Be no longer than 4 paragraphs
    3. Highlight key achievements and experiences from the updated resume
    4. Align with the responsibilities and qualifications in the job description
    5. Reflect the applicant's enthusiasm and fit for the role
    6. End with a confident and polite closing statement

    ---

    ### Output Format (JSON):
    ```json
    {
    "cover_letter": "<final cover letter text>"
    }
    ```
    ---

    ### Input:

    **Updated Resume:**

    """
        + updated_resume
        + """
    **Target Job Description:**

    """
        + job_description_text
    )

    # Depending on the selected model, call the appropriate function to generate the cover letter.
    if model == "openai":
        # Get response from OpenAI API
        updated_cover_letter = openai_generate(prompt, temperature=0.7, response_format=CoverLetterOutput)
    elif model == "gemini":
        # Get response from Gemini API
        updated_cover_letter = gemini_generate(prompt, temperature=0.5)
    else:
        # Raise an error if an invalid model name is provided.
        raise ValueError(f"Invalid model: {model}")

    # Return the generated cover letter as a dictionary.
    return updated_cover_letter


In [15]:
# Call the generate_cover_letter function with the provided job description and updated resume.
updated_cover_letter = generate_cover_letter(job_description_text, updated_resume_json.updated_resume, model="openai")

# Display the generated cover letter in Markdown format.
print_markdown(updated_cover_letter.cover_letter)

Dear Hiring Manager,

I am writing to express my interest in the Digital Marketing Specialist position at BrightWave Digital Agency, as advertised. With over two years of hands-on experience in digital marketing, particularly in planning and executing campaigns across SEO, SEM, social media, and email platforms, I am excited about the opportunity to contribute to your team.

In my current role at Brewster Coffee Co., I successfully assisted in increasing online engagement by 30% through strategic digital marketing initiatives. By leveraging Google Analytics, I improved campaign ROI by 15% through detailed performance reports and data-driven insights. My experience in keyword research and content optimization led to a 20% increase in search engine visibility, aligning closely with the responsibilities outlined in your job description.

I am particularly drawn to the collaborative environment at BrightWave Digital Agency, where I can utilize my skills in social media management and content coordination to drive community engagement and brand consistency. My proficiency with Adobe Creative Suite and basic knowledge of CRM systems like HubSpot further complement your requirements, ensuring that I can effectively support campaign materials and data interpretation.

I am enthusiastic about the possibility of bringing my expertise in digital marketing to BrightWave Digital Agency and contributing to the success of your client campaigns. Thank you for considering my application. I look forward to the opportunity to discuss how I can contribute to your team.

Sincerely,

Jessica Brown

Now that we have all the building blocks, let's create a single function, `run_resume_rocket`, that takes the original resume and job description text and performs the entire workflow: gap analysis, resume tailoring with diff tracking, and cover letter generation. This makes the tool much easier to reuse.

In [16]:
def run_resume_rocket(resume_text: str, job_description_text: str) -> tuple[str, str]:
    """
    Run the resume rocket workflow.

    Args:
        resume_text (str): The candidate's resume text.
        job_description_text (str): The job description text.

    Returns:
        tuple: A tuple containing the updated resume and cover letter.
    """
    # Analyze the candidate's resume against the job description using OpenAI's model.
    # This function will return a structured analysis highlighting gaps and strengths.
    gap_analysis_openai = analyze_resume_against_job_description(job_description_text, 
                                                                 resume_text, 
                                                                 model="openai")

    # Display the gap analysis results in Markdown format for better readability.
    print_markdown(gap_analysis_openai)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Generate an updated resume based on the job description, original resume, and gap analysis.
    # This function will return a JSON-like object containing the updated resume and a diff markdown.
    updated_resume_json = generate_resume(job_description_text, 
                                          resume_text, 
                                          gap_analysis_openai, 
                                          model = "openai")

    # Display the diff markdown which shows the changes made to the resume.
    print_markdown(updated_resume_json.diff_markdown)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Display the updated resume in Markdown format.
    print_markdown(updated_resume_json.updated_resume)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Generate a cover letter based on the job description and the updated resume.
    # This function will return the generated cover letter.
    updated_cover_letter = generate_cover_letter(
        job_description_text, updated_resume_json.updated_resume, model="openai"
    )

    # Display the generated cover letter in Markdown format.
    print_markdown(updated_cover_letter.cover_letter)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    # Return the updated resume and the generated cover letter as a tuple.
    return updated_resume_json.updated_resume, updated_cover_letter.cover_letter



In [17]:
# Call the run_resume_rocket function with the provided resume and job description texts.
resume, cover_letter = run_resume_rocket(resume_text, job_description_text)

**1. Key Requirements from Job Description:**

- Bachelor's degree in Marketing, Communications, or similar.
- 2+ years of digital marketing experience.
- Familiarity with SEO, SEM, Google Analytics, and social media.
- Ability to interpret basic marketing data.
- Good communication and writing skills.
- Knowledge of CRM systems (e.g., HubSpot) helpful.
- Experience with Adobe Creative Suite is beneficial.
- Responsibilities include planning and executing digital marketing campaigns (SEO, SEM, social media, email).
- Use of Google Analytics to measure performance and prepare basic reports.
- Support social media management tasks, including content scheduling and community engagement.
- Perform keyword research and assist in optimizing content for SEO.
- Work with designers to coordinate campaign materials.
- Stay informed about current digital marketing trends.

**2. Relevant Experience in Resume:**

- Holds a Bachelor of Commerce in Marketing.
- 2 years of experience as a Marketing Assistant at Brewster Coffee Co.
- Assisted with digital marketing campaigns including email and social media.
- Created blog posts and managed social media accounts, increasing follower numbers.
- Conducted market research and competitor analysis.
- Skills in Digital Marketing (SEO basics, Email Marketing) and Social Media Tools (Hootsuite, Buffer).
- Basic knowledge of Adobe Photoshop.

**3. Gaps/Mismatches:**

- The resume does not mention experience with SEM or Google Analytics, which are specified in the job description.
- No mention of CRM systems experience (e.g., HubSpot).
- While SEO basics are mentioned, there is no specific mention of performing keyword research or optimizing content for SEO.
- The resume does not explicitly state experience in working with designers to coordinate campaign materials.
- No mention of keeping informed about current digital marketing trends.
- Lack of explicit experience in interpreting marketing data.

**4. Potential Strengths:**

- Experience in creating content, which could be a valuable asset for social media management and content scheduling.
- Effective social media management skills demonstrated by growing follower numbers.
- Experience in conducting market research and competitor analysis, which can provide insights into digital marketing strategies.
- Familiarity with tools like Hootsuite and Buffer, which are useful for social media management.
- Basic knowledge of Adobe Photoshop, which might assist in coordinating with designers for campaign materials.


--------------------------------
--------------------------------



**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
<span style="color:red;text-decoration:line-through">Marketing professsional with 2 years of experience assisting in digital campaigns, content creation, and social media activities. Comfortable handling multiple tasks and providing general marketing support.</span><span style="color:green">Digital Marketing Specialist with over 2 years of experience in executing and managing digital campaigns, including SEO, SEM, social media, and email marketing. Skilled in utilizing Google Analytics for performance measurement and reporting, with a strong ability to interpret marketing data and optimize content for SEO. Adept at coordinating with designers to create compelling campaign materials and staying informed on the latest digital marketing trends.</span>

**Experience**

**Marketing Assistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- <span style="color:red;text-decoration:line-through">Assisted with digital marketing campaigns including email and social media.</span><span style="color:green">Assisted in planning and executing digital marketing campaigns, including email and social media, resulting in a 20% increase in engagement.</span>
- <span style="color:green">Utilized Google Analytics to measure campaign performance and prepare basic reports, enhancing data-driven decision-making.</span>
- <span style="color:red;text-decoration:line-through">Created blog posts and social media updates to improve audience engagement.</span><span style="color:green">Supported social media management tasks, including content scheduling and community engagement, leading to a 30% growth in followers.</span>
- <span style="color:red;text-decoration:line-through">Managed social media accounts and grew follower numbers.</span>
- <span style="color:red;text-decoration:line-through">Supported coordination of marketing events.</span><span style="color:green">Conducted keyword research and assisted in optimizing content for SEO, improving organic search rankings.</span>
- <span style="color:red;text-decoration:line-through">Conducted market research and competitor analysis.</span><span style="color:green">Collaborated with designers to coordinate the creation of marketing materials for various campaigns.</span>
- <span style="color:green">Regularly updated knowledge on current digital marketing trends to implement innovative strategies.</span>

**Skills**
- <span style="color:green">Digital Marketing (SEO, SEM, Email Marketing)</span>
- <span style="color:green">Google Analytics, </span>Social Media Tools (Hootsuite, Buffer)
- Microsoft Office Suite, Google Workspace
- <span style="color:green">Basic knowledge of Adobe Photoshop</span>
- <span style="color:green">Familiar with CRM systems (e.g., HubSpot)</span>

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021


--------------------------------
--------------------------------



**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Digital Marketing Specialist with over 2 years of experience in executing and managing digital campaigns, including SEO, SEM, social media, and email marketing. Skilled in utilizing Google Analytics for performance measurement and reporting, with a strong ability to interpret marketing data and optimize content for SEO. Adept at coordinating with designers to create compelling campaign materials and staying informed on the latest digital marketing trends.

**Experience**

**Marketing Assistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted in planning and executing digital marketing campaigns, including email and social media, resulting in a 20% increase in engagement.
- Utilized Google Analytics to measure campaign performance and prepare basic reports, enhancing data-driven decision-making.
- Supported social media management tasks, including content scheduling and community engagement, leading to a 30% growth in followers.
- Conducted keyword research and assisted in optimizing content for SEO, improving organic search rankings.
- Collaborated with designers to coordinate the creation of marketing materials for various campaigns.
- Regularly updated knowledge on current digital marketing trends to implement innovative strategies.

**Skills**
- Digital Marketing (SEO, SEM, Email Marketing)
- Google Analytics, Social Media Tools (Hootsuite, Buffer)
- Microsoft Office Suite, Google Workspace
- Basic knowledge of Adobe Photoshop
- Familiar with CRM systems (e.g., HubSpot)

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021


--------------------------------
--------------------------------



Dear Hiring Manager,

I am writing to express my interest in the Digital Marketing Specialist position at BrightWave Digital Agency. With over two years of experience in digital marketing, including hands-on expertise in SEO, SEM, social media, and email marketing, I am excited about the opportunity to contribute to your team and help manage diverse client campaigns.

In my current role as a Marketing Assistant at Brewster Coffee Co., I have successfully assisted in planning and executing digital marketing campaigns that have increased engagement by 20%. My experience in utilizing Google Analytics to measure campaign performance has enhanced my ability to make data-driven decisions and optimize content for SEO, leading to improved organic search rankings. I am particularly proud of my role in growing our social media following by 30% through effective content scheduling and community engagement strategies.

I am well-versed in coordinating with designers to create compelling marketing materials and pride myself on staying up-to-date with the latest digital marketing trends. My academic background in marketing from Ryerson University, coupled with practical experience using tools like Hootsuite, Buffer, and CRM systems, aligns closely with the qualifications you seek.

I am eager to bring my skills in digital marketing and data analysis to BrightWave Digital Agency. I am confident that my proactive approach and enthusiasm for innovative marketing strategies will be a valuable asset to your team. Thank you for considering my application. I look forward to the opportunity to discuss how I can contribute to your agency's success.

Sincerely,

Jessica Brown


--------------------------------
--------------------------------

